# 4.15 · 稳健回归 / Robust Regression

> **课程定位 / Where this fits**
> 第 15 课，**Part 4 · 监督学习：回归**。
> Lesson 15, **Part 4 · Supervised Regression**.
>
> 普通 OLS 用平方损失，对**异常值极度敏感**——几个离群点就能把整条回归线拉歪（4.2 的影响点）。**稳健回归**通过改损失或显式识别离群点，让模型在有异常值时仍能拟合出可靠趋势。当你的数据里有无法删除的离群点时，这是 OLS 的标准替代。
> OLS uses squared loss and is **extremely sensitive to outliers** — a few can tilt the whole line (the influential points from 4.2). **Robust regression** changes the loss or explicitly identifies outliers so the model fits a reliable trend despite them. The standard alternative to OLS when undeleteable outliers exist.
>
> 💼 **实战/面试视角**："OLS 为什么怕异常值 / Huber/RANSAC 是什么 / 击穿点" 偏统计/稳健建模。
> 💼 **Practical/interview angle:** "why OLS fears outliers / Huber/RANSAC / breakdown point" — robust modeling.

> 💡 **面试相关 / Interview-relevant**
> - "OLS 为什么对异常值敏感"（出镜率 ★★★★，平方放大大误差）
> - "Huber loss 是什么 / 怎么稳健"（★★★★）
> - "RANSAC 原理"（★★★★）
> - "击穿点(breakdown point)是什么"（★★★）

---

## 学习目标 / Learning Objectives

1. 看清 OLS 被少量异常值拉偏的程度。
   See how badly a few outliers tilt OLS.
2. 理解 **Huber loss**（小误差平方、大误差线性）。
   Understand the Huber loss (squared for small, linear for large errors).
3. 用 **RANSAC** 应对极端污染（自动识别内点）。
   Use RANSAC for extreme contamination (auto-identify inliers).
4. 了解 **Theil-Sen** 及**击穿点**概念。
   Know Theil-Sen and the breakdown-point concept.
5. 横向比较各方法的稳健性。
   Compare methods' robustness across contamination levels.

## 目录 / TOC
1. [先建直觉：OLS 为何怕异常值 ⭐](#1)
2. [Huber 回归 ⭐](#2)
3. [RANSAC：极端污染 ⭐](#3)
4. [Theil-Sen + 击穿点 ⭐](#4)
5. [稳健性横向对比 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：OLS 为何怕异常值 ⭐ / Why OLS Fears Outliers

OLS 最小化**平方**误差。平方有个副作用：一个残差为 10 的点，损失是 100；残差为 1 的点损失只有 1。所以**一个大误差的权重相当于 100 个小误差**——回归线会被少数离群点"拽"过去，以减小它们巨大的平方损失。结果是仅 5% 的异常值就能显著拉偏斜率。
OLS minimizes **squared** errors. Squaring has a side effect: a point with residual 10 contributes loss 100, while residual 1 contributes only 1. So **one large error counts like 100 small ones** — the line gets "dragged" toward a few outliers to reduce their huge squared loss. Just 5% outliers can noticeably tilt the slope.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

n = 100
x = np.sort(rng.uniform(0, 10, n))
y = 2*x + 1 + rng.normal(0, 1, n)          # 真实斜率 2.0
y_dirty = y.copy(); y_dirty[-5:] = y[-5:] - 40   # 注入 5 个异常值(把右端拉低)
X = x.reshape(-1, 1)

ols_clean = LinearRegression().fit(X, y)
ols_dirty = LinearRegression().fit(X, y_dirty)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x[:-5], y_dirty[:-5], alpha=0.4, s=15, label="正常点 normal")
ax.scatter(x[-5:], y_dirty[-5:], c="red", s=60, label="异常值 outliers(5个)")
xp = np.linspace(0, 10, 50).reshape(-1, 1)
ax.plot(xp, ols_clean.predict(xp), "g--", lw=2, label=f"OLS(无异常) 斜率={ols_clean.coef_[0]:.2f}")
ax.plot(xp, ols_dirty.predict(xp), "r-", lw=2, label=f"OLS(含异常) 斜率={ols_dirty.coef_[0]:.2f}")
ax.legend(fontsize=9); ax.set_title("仅 5 个异常值(5%)就把 OLS 斜率从 2.0 拉偏")
plt.tight_layout(); plt.show()
print(f"真实斜率 2.0; 无异常 OLS={ols_clean.coef_[0]:.2f}; 含 5% 异常 OLS={ols_dirty.coef_[0]:.2f}")
print("→ 仅 5% 异常值就显著拉偏 — 平方损失对大误差过度敏感")


<a id="2"></a>
## 2. Huber 回归 ⭐ / Huber Regression

**Huber loss** 是 OLS 和绝对损失的折中：误差小（$|r|\le\delta$）时用**平方**（保持 OLS 的高效与平滑），误差大（$|r|>\delta$）时切换成**线性**（不让大误差获得过大权重）。这样异常值只受**线性**惩罚，拉不动回归线，而正常点仍享受平方损失的好处。是最常用的稳健回归。
**Huber loss** blends OLS and absolute loss: for small errors ($|r|\le\delta$) it's **squared** (keeping OLS's efficiency and smoothness); for large errors ($|r|>\delta$) it switches to **linear** (so big errors don't get outsized weight). Outliers thus get only **linear** penalty and can't drag the line, while normal points keep squared-loss benefits. The most commonly used robust regressor.


In [ ]:
from sklearn.linear_model import HuberRegressor

r = np.linspace(-4, 4, 300); delta = 1.0
huber = np.where(np.abs(r) <= delta, 0.5*r**2, delta*(np.abs(r)-0.5*delta))   # Huber loss 定义
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(r, 0.5*r**2, "r--", label="平方损失 OLS"); axes[0].plot(r, huber, "b-", lw=2, label=f"Huber (δ={delta})")
axes[0].plot(r, np.abs(r), "g:", label="绝对损失 MAE"); axes[0].axvspan(-delta, delta, alpha=0.1, color="blue")
axes[0].legend(); axes[0].set_xlabel("残差 r"); axes[0].set_title("Huber: |r|<δ 平方, |r|>δ 线性")

huber_reg = HuberRegressor().fit(X, y_dirty)        # Huber 回归拟合含异常数据
axes[1].scatter(x[:-5], y_dirty[:-5], alpha=0.4, s=15); axes[1].scatter(x[-5:], y_dirty[-5:], c="red", s=50)
xp = np.linspace(0,10,50).reshape(-1,1)
axes[1].plot(xp, ols_dirty.predict(xp), "r-", lw=2, label=f"OLS 斜率={ols_dirty.coef_[0]:.2f}")
axes[1].plot(xp, huber_reg.predict(xp), "b-", lw=2, label=f"Huber 斜率={huber_reg.coef_[0]:.2f}")
axes[1].plot(xp, ols_clean.predict(xp), "g--", lw=1.5, label="真实(无异常)")
axes[1].legend(fontsize=8); axes[1].set_title("Huber 几乎不受异常值影响")
plt.tight_layout(); plt.show()
print(f"含异常时: OLS 斜率={ols_dirty.coef_[0]:.2f}, Huber 斜率={huber_reg.coef_[0]:.2f} (真实 2.0)")
print("Huber 对异常值的大残差只用线性惩罚 → 它们拉不动回归线 → 斜率接近真实")


<a id="3"></a>
## 3. RANSAC：极端污染 ⭐ / RANSAC for Extreme Contamination

当异常值高达 40%-50% 时，连 Huber 都扛不住。**RANSAC**（随机抽样一致）换了个策略：**反复随机抽一小撮点拟合模型，看有多少其它点'同意'（落在容差内）**，保留'同意者(内点)'最多的那个模型，彻底忽略外点。它能在**接近一半数据是垃圾**的情况下仍准确恢复趋势。
When outliers reach 40%-50%, even Huber fails. **RANSAC** (RANdom SAmple Consensus) takes a different approach: **repeatedly fit a model on a small random subset and count how many other points "agree" (fall within tolerance)**, keeping the model with the most agreeing inliers and discarding outliers entirely. It recovers the trend even when **nearly half the data is garbage**.


In [ ]:
from sklearn.linear_model import RANSACRegressor

y_extreme = y.copy()
out_idx = rng.choice(n, 40, replace=False)
y_extreme[out_idx] = rng.uniform(-30, 50, 40)      # 40% 完全随机的异常! / 40% contamination

ransac = RANSACRegressor(random_state=0).fit(X, y_extreme)
ols_ext = LinearRegression().fit(X, y_extreme)
inlier_mask = ransac.inlier_mask_                   # RANSAC 判定的内点掩码

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x[inlier_mask], y_extreme[inlier_mask], alpha=0.5, s=15, label="RANSAC 内点 inliers")
ax.scatter(x[~inlier_mask], y_extreme[~inlier_mask], c="red", alpha=0.5, s=15, label="RANSAC 外点 outliers")
xp = np.linspace(0,10,50).reshape(-1,1)
ax.plot(xp, ols_ext.predict(xp), "r-", lw=2, label=f"OLS 斜率={ols_ext.coef_[0]:.2f}")
ax.plot(xp, ransac.predict(xp), "b-", lw=2, label=f"RANSAC 斜率={ransac.estimator_.coef_[0]:.2f}")
ax.legend(fontsize=8); ax.set_title("40% 异常值! OLS 崩溃, RANSAC 自动识别内点并准确拟合")
plt.tight_layout(); plt.show()
print(f"40% 异常下: OLS 斜率={ols_ext.coef_[0]:.2f}(崩), RANSAC={ransac.estimator_.coef_[0]:.2f}(真实 2.0!)")
print(f"RANSAC 自动把 {(~inlier_mask).sum()} 个点判为外点并忽略 → 高击穿点")


<a id="4"></a>
## 4. Theil-Sen + 击穿点 ⭐ / Theil-Sen & Breakdown Point

**Theil-Sen** 用一个优雅的中位数思想：算**所有点对**连线的斜率，取它们的**中位数**作为最终斜率。因为中位数对极端值不敏感，少数异常点对产生的极端斜率会被中位数自动过滤掉。
**Theil-Sen** uses an elegant median idea: compute the slope of **every pair of points** and take their **median** as the final slope. Since the median ignores extremes, the wild slopes from a few outlier pairs are automatically filtered out.

**击穿点(breakdown point)**（面试概念）：一个估计器能容忍的**最大异常值比例**，超过它估计就彻底失效。OLS 的击穿点是 **0**（一个异常值就能拉到任意远）；Huber 较高；Theil-Sen 约 **29%**；RANSAC 可超过 **50%**。这是衡量稳健性的标准指标。
**Breakdown point** (an interview concept): the **maximum fraction of outliers** an estimator tolerates before it fails completely. OLS's is **0** (one outlier can move it arbitrarily); Huber's is higher; Theil-Sen's is ~**29%**; RANSAC's can exceed **50%**. The standard measure of robustness.


In [ ]:
from sklearn.linear_model import TheilSenRegressor

theil = TheilSenRegressor(random_state=0).fit(X, y_dirty)
print("5% 异常下各方法斜率(真实 2.0):")
print(f"  OLS:       {ols_dirty.coef_[0]:.3f}  (被拉偏)")
print(f"  Huber:     {HuberRegressor().fit(X, y_dirty).coef_[0]:.3f}")
print(f"  Theil-Sen: {theil.coef_[0]:.3f}")
print(f"  RANSAC:    {RANSACRegressor(random_state=0).fit(X, y_dirty).estimator_.coef_[0]:.3f}")
print("\nTheil-Sen = 所有点对斜率的中位数; 异常点对的极端斜率被中位数过滤掉")


<a id="5"></a>
## 5. 稳健性横向对比 + 小结 ⭐ / Robustness Showdown & Summary

把污染比例从 0% 加到 50%，看各方法估计的斜率（真实 2.0）何时崩。结论与击穿点理论一致：**OLS 一污染就偏；Huber 抗中度污染；Theil-Sen 撑到约 29%；RANSAC 最强（可 >50%）**。
Ramp contamination from 0% to 50% and watch when each method's slope estimate (true 2.0) breaks. The result matches breakdown-point theory: **OLS tilts immediately; Huber resists moderate; Theil-Sen lasts to ~29%; RANSAC strongest (>50%)**.


In [ ]:
methods = {"OLS": LinearRegression(), "Huber": HuberRegressor(),
           "Theil-Sen": TheilSenRegressor(random_state=0), "RANSAC": RANSACRegressor(random_state=0)}
contam = [0.0, 0.1, 0.2, 0.35, 0.5]
print(f"{'方法 method':<12} " + " ".join(f"{int(c*100):>5}%" for c in contam) + "   (估计斜率, 真实=2.0)")
print("-" * 56)
for name, model in methods.items():
    row = []
    for c in contam:
        yd = y.copy()
        if c > 0:
            idx = rng.choice(n, int(n*c), replace=False)
            yd[idx] = rng.uniform(-30, 50, len(idx))      # 注入 c 比例的随机异常
        try:
            model.fit(X, yd)
            slope = model.estimator_.coef_[0] if name == "RANSAC" else model.coef_[0]
            row.append(f"{slope:>6.2f}")
        except Exception:
            row.append("  fail")
    print(f"{name:<12} " + " ".join(row))
print("\nOLS: 一污染就偏; Huber: 中度污染稳; Theil-Sen: ~29% 击穿; RANSAC: 最高(>50%)")
print("击穿点排序: OLS(0) < Huber < Theil-Sen(~29%) < RANSAC(可>50%)")


```
OLS 怕异常值: 平方损失把大误差放大(残差10→损失100), 5% 异常就拉偏; 击穿点=0
Huber: |r|<δ 平方, |r|>δ 线性 → 异常值只受线性罚, 拉不动线; 最常用稳健回归
RANSAC: 反复随机抽样拟合, 保留内点最多的模型, 忽略外点; 抗极端污染(>50%)
Theil-Sen: 所有点对斜率的中位数; 中位数过滤极端斜率; 击穿点~29%
击穿点 = 能容忍的最大异常比例; OLS(0)<Huber<Theil-Sen(29%)<RANSAC(>50%)
适用: 数据有不可删的离群点时用稳健回归; 先查是不是数据错误(3.3)
```

### 💡 面试速查 / Interview cheat-sheet
1. **OLS 怕异常值**: 平方损失放大大误差; 击穿点 = 0。
   OLS fears outliers: squared loss amplifies them; breakdown point = 0.
2. **Huber**: 小误差平方、大误差线性 → 异常值拉不动线。
   Huber: squared for small, linear for large → outliers can't drag the line.
3. **RANSAC**: 随机抽样找内点共识, 抗极端污染(>50%)。
   RANSAC: random-sample inlier consensus, handles extreme contamination.
4. **Theil-Sen**: 点对斜率的中位数; 击穿点~29%。
   Theil-Sen: median of pairwise slopes; ~29% breakdown.
5. **击穿点**衡量稳健性: OLS<Huber<Theil-Sen<RANSAC。
   Breakdown point measures robustness: OLS<Huber<Theil-Sen<RANSAC.

### 下一节 / Next
**4.16 等张回归**——Part 4 收官: 当你只知道关系是单调的(但不知具体形状), 等张回归拟合一个单调阶梯; 最重要的工业用途是概率校准。
**4.16 Isotonic Regression** — Part 4 finale: when you only know a relationship is monotonic (but not its shape), isotonic regression fits a monotonic staircase; its key industrial use is probability calibration.
